<a href="https://colab.research.google.com/github/Dra001ven/Team-6_Healthcare-hackathon/blob/main/eagles.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
# Data handling
import pandas as pd
import numpy as np

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns

# Machine learning
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.ensemble import IsolationForest
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

# Settings
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)
from google.colab import drive
print("All libraries imported successfully")

All libraries imported successfully


In [17]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [23]:
ct = pd.read_csv('/content/drive/MyDrive/contact_tracing.csv')
mob = pd.read_csv('/content/drive/MyDrive/mobility.csv')
vitals = pd.read_csv('/content/drive/MyDrive/vitals.csv')

print(f"contact_tracing: {ct.shape[0]:,} rows, {ct.shape[1]} columns")
print(f"mobility:        {mob.shape[0]:,} rows, {mob.shape[1]} columns")
print(f"vitals:          {vitals.shape[0]:,} rows, {vitals.shape[1]} columns")

contact_tracing: 119,338 rows, 9 columns
mobility:        50,737 rows, 11 columns
vitals:          2,307 rows, 12 columns


In [24]:
print("=== FIRST 5 ROWS ===")
print(ct.head())
print()
print("=== COLUMN INFO ===")
print(ct.info())
print()
print("=== MISSING VALUES ===")
print(ct.isnull().sum())
print()
print("=== PROXIMITY CATEGORIES ===")
print(ct['proximity'].value_counts())
print()
print(f"Unique users: {ct['user_id'].nunique()}")
print(f"Unique MACs: {ct['mac'].nunique()}")
print(f"Unscored RSSI (-1): {(ct['rssi'] == -1).sum()} ({(ct['rssi'] == -1).mean()*100:.1f}%)")

=== FIRST 5 ROWS ===
  user_id        date      time  latitude  longitude    geohash     mac  rssi  \
0    U002  2023-10-19  18:06:01      7.68       6.42  s1kedn1tq  D13941    -1   
1    U002  2023-10-19  18:06:01      7.68       6.42  s1kedn1tq  D13227     8   
2    U002  2023-10-19  18:06:01      7.68       6.42  s1kedn1tq  D21917    -1   
3    U002  2023-10-19  18:06:01      7.68       6.42  s1kedn1tq  D19180    -1   
4    U003  2023-10-20  06:12:13      5.37       7.00  s0uyx62pp  D05480     7   

  proximity  
0  unscored  
1     close  
2  unscored  
3  unscored  
4     close  

=== COLUMN INFO ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 119338 entries, 0 to 119337
Data columns (total 9 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   user_id    119338 non-null  object 
 1   date       119338 non-null  object 
 2   time       119338 non-null  object 
 3   latitude   119338 non-null  float64
 4   longitude  119338 non-null

In [22]:
print("=== FIRST 5 ROWS ===")
print(vitals.head())
print()
print("=== TEMPERATURE STATUS ===")
print(vitals['temp_status'].value_counts())
print()
print("=== HEART RATE STATUS ===")
print(vitals['hr_status'].value_counts())
print()
print(f"Unique devices: {vitals['device_id'].nunique()}")

=== FIRST 5 ROWS ===
  device_id        date      time  latitude  longitude    geohash  \
0      D009  2024-05-13  06:56:50      7.30       5.14  s179sctvp   
1      D009  2024-05-13  06:57:00      7.30       5.14  s179sctus   
2      D009  2024-05-13  06:57:20      7.30       5.14  s179sctew   
3      D009  2024-05-13  07:16:00      7.30       5.13  s179scc4h   
4      D009  2024-05-13  07:17:25      7.30       5.13  s179scc1q   

   temperature temp_status  heartbeat hr_status  movement  battery  
0        31.00         low      96.00    normal      0.00   100.00  
1        31.00         low     110.00      high      0.00   100.00  
2        30.00         low      76.00    normal      1.00   100.00  
3        34.00         low      49.00       low      0.00   100.00  
4        34.00         low      43.00       low      0.00   100.00  

=== TEMPERATURE STATUS ===
temp_status
low       2002
normal     168
high       137
Name: count, dtype: int64

=== HEART RATE STATUS ===
hr_status
no

In [26]:
ct['timestamp'] = pd.to_datetime(ct['date'].astype(str) + ' ' + ct['time'].astype(str))
ct['date'] = ct['timestamp'].dt.date
ct['hour'] = ct['timestamp'].dt.hour
ct['month'] = ct['timestamp'].dt.month
ct['rssi_clean'] = ct['rssi'].replace(-1, np.nan)

before = len(ct)
ct.drop_duplicates(inplace=True)
after = len(ct)

print(f"Duplicates removed: {before - after}")
print(f"Rows remaining: {after:,}")
print("Cleaning complete.")

Duplicates removed: 4
Rows remaining: 119,334
Cleaning complete.
